# 10. SentenceBERT model with all-MiniLM-L6-v2

This notebook evaluates a frozen SentenceBERT representation with a logistic-regression classifier. It follows the repository model workflow: grouped validation, baseline tuning on a 10,000-row sample, backward feature-block elimination, full-training tuning with the selected blocks, final fitting, and untouched test evaluation.

The movie-disjoint split is the primary generalisation experiment. The review-text-grouped split is retained for comparison. SentenceBERT embeddings are computed without using spoiler labels; classifier tuning and feature selection use training data only.


## 1. Setup

Install `sentence-transformers` and a compatible PyTorch build in the notebook environment before running this notebook. The encoder is frozen initially, so only the downstream classifier is trained. The default model is `all-MiniLM-L6-v2`, chosen as a practical baseline; replace `MODEL_NAME` for a controlled model comparison.


In [1]:
from pathlib import Path
import sys
import json
import gc
import numpy as np
import pandas as pd
import torch
from scipy.sparse import csr_matrix, hstack
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import ParameterGrid, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

try:
    from sentence_transformers import SentenceTransformer
except ImportError as exc:
    raise ImportError("Install sentence-transformers in this notebook environment before running it.") from exc

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start Jupyter from the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_cleaning import clean_data
from src.metrics import evaluate
from src.prepare_features import prepare_features
from src.split_data import create_train_test_split

RANDOM_STATE = 67
N_FOLDS = 5
SAMPLE_SIZE = 10_000
MODEL_NAME = "all-MiniLM-L6-v2"
BATCH_SIZE = 64
MAX_ITER = 1_000
FEATURE_BLOCKS = [
    "review_text", "review_summary", "plot_summary", "plot_synopsis",
    "rating", "genre", "duration", "days_since_release",
]
TEXT_BLOCKS = ["review_text", "review_summary", "plot_summary", "plot_synopsis"]
NUMERIC_BLOCKS = ["rating", "duration", "days_since_release"]
CATEGORICAL_BLOCKS = ["genre"]
MODEL_GRID = {"C": [0.1, 1.0, 10.0], "class_weight": [None, "balanced"]}
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "sentencebert_comparison"


## 2. Load grouped splits and prepare inputs

The existing splitter prevents duplicate review text from crossing partitions and keeps movie-connected rows together for the movie-disjoint condition. No identifiers are included as model features.


In [2]:
cleaned_reviews_path = PROJECT_ROOT / "data" / "cleaned" / "IMDB_reviews.json"
cleaned_movies_path = PROJECT_ROOT / "data" / "cleaned" / "IMDB_movie_details.json"
if not cleaned_reviews_path.exists() or not cleaned_movies_path.exists():
    clean_data()
_, standard_train_reviews, standard_test_reviews = create_train_test_split("review_text")

movies = pd.read_json(cleaned_movies_path, lines=True)

split_reviews = {
    "review_text_grouped": (standard_train_reviews, standard_test_reviews, "review_text"),
}
prepared = {}
for name, (train_reviews, test_reviews, group_by) in split_reviews.items():
    train = prepare_features(train_reviews, movies)
    test = prepare_features(test_reviews, movies)
    train["is_spoiler"] = train_reviews["is_spoiler"].to_numpy()
    test["is_spoiler"] = test_reviews["is_spoiler"].to_numpy()
    train["movie_id"] = train_reviews["movie_id"].to_numpy()
    test["movie_id"] = test_reviews["movie_id"].to_numpy()
    prepared[name] = (train, test, group_by)
print({name: (len(train), len(test), group) for name, (train, test, group) in prepared.items()})


Train: 459,130 (80.00%), spoiler rate 26.30%
Test:  114,783 (20.00%), spoiler rate 26.30%
{'review_text_grouped': (459130, 114783, 'review_text')}


## 3. Encode text and build feature blocks

Each text field receives its own SentenceBERT embedding so that backward elimination can remove interpretable blocks. Metadata preprocessing is fitted separately inside each validation fold. The frozen encoder may encode the outer test rows because it is not trained on project labels; the classifier and all preprocessing choices remain training-only.


In [3]:
encoder = SentenceTransformer(MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

def add_embeddings(data, cache_name):
    result = data.copy()
    embeddings = {}
    for block in TEXT_BLOCKS:
        cache_path = OUTPUT_DIR / f"{cache_name}_{block}.npy"
        values = result[block].fillna("").astype(str).tolist()
        if cache_path.exists():
            embedding = np.load(cache_path, mmap_mode="r")
        else:
            embedding = encoder.encode(values, batch_size=BATCH_SIZE, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)
            embedding = np.asarray(embedding, dtype=np.float32)
            OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            np.save(cache_path, embedding)
        embedding = np.asarray(embedding, dtype=np.float32)
        if len(embedding) != len(result):
            raise ValueError(f"Embedding length mismatch for {cache_name}/{block}.")
        embeddings[block] = embedding
    return result, embeddings

def make_metadata_transformer(selected_blocks):
    numeric_blocks = [block for block in selected_blocks if block in NUMERIC_BLOCKS]
    categorical_blocks = [block for block in selected_blocks if block in CATEGORICAL_BLOCKS]
    transformers = []
    if numeric_blocks:
        transformers.append(("numeric", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_blocks))
    if categorical_blocks:
        transformers.append(("categorical", Pipeline([("impute", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_blocks))
    return ColumnTransformer(transformers, remainder="drop")

def prepare_metadata_input(data, metadata_blocks):
    metadata = data.copy()
    if "genre" in metadata_blocks:
        def normalize_genre(value):
            if isinstance(value, (list, tuple, set, np.ndarray)):
                return ", ".join(str(item) for item in value)
            if pd.isna(value):
                return ""
            return str(value)
        metadata["genre"] = metadata["genre"].map(normalize_genre)
    return metadata

def make_matrix(data, embeddings, selected_blocks, metadata_transformer=None, fit_metadata=False):
    matrices = []
    text_blocks = [block for block in selected_blocks if block in TEXT_BLOCKS]
    for block in text_blocks:
        row_indices = data.index.to_numpy()
        matrices.append(csr_matrix(embeddings[block][row_indices], dtype=np.float32))
    metadata_blocks = [block for block in selected_blocks if block in NUMERIC_BLOCKS + CATEGORICAL_BLOCKS]
    if metadata_blocks:
        if metadata_transformer is None:
            metadata_transformer = make_metadata_transformer(metadata_blocks)
        metadata_input = prepare_metadata_input(data, metadata_blocks)
        if fit_metadata:
            metadata_transformer.fit(metadata_input)
        matrices.append(csr_matrix(metadata_transformer.transform(metadata_input)))
    if not matrices:
        raise ValueError("At least one feature block must be selected.")
    return hstack(matrices, format="csr"), metadata_transformer


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 4. Grouped validation and tuning helpers

The helper below supports the required two-stage tuning process. The 10,000-row stage is used only to identify baseline classifier parameters. Feature elimination freezes those parameters. The second tuning stage uses the complete outer training partition and the selected feature blocks.


In [4]:
def grouped_tune(data, embeddings, y, groups, selected_blocks, model_grid, sample_size=None):
    if sample_size is not None and sample_size < len(data):
        rng = np.random.default_rng(RANDOM_STATE)
        sample_indices = rng.choice(len(data), size=sample_size, replace=False)
        data = data.iloc[sample_indices]
        y = np.asarray(y)[sample_indices]
        groups = np.asarray(groups)[sample_indices]
    splitter = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for params in ParameterGrid(model_grid):
        fold_scores = []
        for train_idx, valid_idx in splitter.split(data, y, groups):
            metadata_transformer = make_metadata_transformer(selected_blocks)
            X_train, metadata_transformer = make_matrix(data.iloc[train_idx], embeddings, selected_blocks, metadata_transformer, True)
            X_valid, _ = make_matrix(data.iloc[valid_idx], embeddings, selected_blocks, metadata_transformer, False)
            model = LogisticRegression(max_iter=MAX_ITER, solver="liblinear", random_state=RANDOM_STATE, **params)
            model.fit(X_train, y[train_idx])
            scores = evaluate(y[valid_idx], model.predict(X_valid), model.predict_proba(X_valid)[:, 1])
            scores["average_precision"] = scores.pop("pr_auc")
            fold_scores.append(scores)
            del X_train, X_valid, metadata_transformer, model
        rows.append({**params, "macro_f1_mean": np.mean([s["macro_f1"] for s in fold_scores]), "macro_f1_std": np.std([s["macro_f1"] for s in fold_scores]), "average_precision_mean": np.mean([s["average_precision"] for s in fold_scores])})
    return pd.DataFrame(rows).sort_values("macro_f1_mean", ascending=False).reset_index(drop=True)

def fit_and_score(train_data, test_data, train_embeddings, test_embeddings, y_train, y_test, selected_blocks, params):
    metadata_transformer = make_metadata_transformer(selected_blocks)
    X_train, metadata_transformer = make_matrix(train_data, train_embeddings, selected_blocks, metadata_transformer, True)
    X_test, _ = make_matrix(test_data, test_embeddings, selected_blocks, metadata_transformer, False)
    model = LogisticRegression(max_iter=MAX_ITER, solver="liblinear", random_state=RANDOM_STATE, **params)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]
    scores = evaluate(y_test, predictions, probabilities)
    scores["average_precision"] = scores.pop("pr_auc")
    return model, scores, predictions, probabilities


## 5. Baseline tuning, backward elimination, and final evaluation

Backward elimination uses a tolerance of 0.005 macro-F1 from the best visited configuration. It removes one interpretable feature block at a time and keeps the smallest retained set within that tolerance.


In [5]:
def run_workflow(name, train_data, test_data, train_embeddings, test_embeddings, group_by):
    y_train = train_data["is_spoiler"].to_numpy().astype(int)
    y_test = test_data["is_spoiler"].to_numpy().astype(int)
    groups = train_data[group_by].to_numpy()
    baseline = grouped_tune(train_data, train_embeddings, y_train, groups, FEATURE_BLOCKS, MODEL_GRID, SAMPLE_SIZE)
    baseline_params = {key: baseline.iloc[0][key] for key in MODEL_GRID}
    current = FEATURE_BLOCKS.copy()
    elimination_rows = []
    best_score = -np.inf
    best_blocks = current.copy()
    while len(current) > 1:
        candidates = []
        for removed in current:
            candidate = [block for block in current if block != removed]
            validation = grouped_tune(train_data, train_embeddings, y_train, groups, candidate, {key: [value] for key, value in baseline_params.items()}, SAMPLE_SIZE)
            score = float(validation.iloc[0]["macro_f1_mean"])
            candidates.append((score, candidate, removed))
            elimination_rows.append({"removed": removed, "remaining": candidate, "macro_f1_mean": score})
        score, candidate, removed = max(candidates, key=lambda item: item[0])
        if score > best_score:
            best_score, best_blocks = score, candidate.copy()
        current = candidate
    full_tuning = grouped_tune(train_data, train_embeddings, y_train, groups, best_blocks, MODEL_GRID)
    final_params = {key: full_tuning.iloc[0][key] for key in MODEL_GRID}
    model, test_scores, predictions, probabilities = fit_and_score(train_data, test_data, train_embeddings, test_embeddings, y_train, y_test, best_blocks, final_params)
    result = {"split": name, "best_blocks": best_blocks, "baseline_params": baseline_params, "final_params": final_params, **test_scores}
    return result, baseline, pd.DataFrame(elimination_rows), full_tuning, predictions, probabilities

all_results = []
artifacts = {}
name = "review_text_grouped"
train_data, test_data, group_by = prepared[name]
train_data, train_embeddings = add_embeddings(train_data, f"{name}_train")
test_data, test_embeddings = add_embeddings(test_data, f"{name}_test")
result, baseline, elimination, full_tuning, predictions, probabilities = run_workflow(name, train_data, test_data, train_embeddings, test_embeddings, group_by)
all_results.append(result)
artifacts[name] = (baseline, elimination, full_tuning, predictions, probabilities)
test_movie_ids = test_data["movie_id"].to_numpy()
del train_data, test_data, train_embeddings, test_embeddings
gc.collect()

results_df = pd.DataFrame(all_results)
display(results_df)


,split,best_blocks,baseline_params,final_params,accuracy,precision,recall,f1,macro_f1,tn,fp,fn,tp,average_precision
0,review_text_grouped,"[review_text, review_summary, plot_summary, pl...","{'C': 0.1, 'class_weight': 'balanced'}","{'C': 10.0, 'class_weight': 'balanced'}",0.674691,0.426345,0.686036,0.525877,0.639141,56735,27863,9477,20708,0.51141


## 6. Save outputs

Saved artifacts include baseline tuning, elimination paths, full-data tuning, final metrics, and test predictions. The test predictions are generated only after feature selection and final hyperparameter tuning.


In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_df.to_json(OUTPUT_DIR / "test_metrics.json", orient="records", indent=2)
for name, (baseline, elimination, full_tuning, predictions, probabilities) in artifacts.items():
    baseline.to_csv(OUTPUT_DIR / f"{name}_baseline_tuning.csv", index=False)
    elimination.to_json(OUTPUT_DIR / f"{name}_feature_elimination.json", orient="records", indent=2)
    full_tuning.to_csv(OUTPUT_DIR / f"{name}_full_tuning.csv", index=False)
    test_reviews = prepared[name][1]
    pd.DataFrame({"movie_id": test_reviews["movie_id"].to_numpy(), "is_spoiler": test_reviews["is_spoiler"].to_numpy(), "prediction": predictions, "probability": probabilities}).to_csv(OUTPUT_DIR / f"{name}_predictions.csv", index=False)
print(f"Saved SentenceBERT outputs to {OUTPUT_DIR}")


Saved SentenceBERT outputs to c:\Users\quanh\OneDrive - National University of Singapore\Y4S1\CS3244\Project\cs3244_group12_imdb_spoilers\outputs\sentencebert_comparison


## 7. Discussions

### Workflow and data split

This notebook evaluates a frozen SentenceBERT encoder with a logistic-regression classifier on the review-text-grouped split. Identical review text is kept in one partition, while the active workflow uses grouped five-fold validation within the training partition. The test set remains untouched until the final evaluation. The separate text embeddings and metadata preprocessing allow feature blocks to be compared without including identifiers such as `movie_id` as model inputs.

### Model-selection findings

The baseline search used a 10,000-row sample and all available feature blocks. Its selected parameters were `C=0.1` with balanced class weights. Backward elimination then evaluated interpretable feature blocks using those parameters. The final full-training tuning selected `C=10.0` with balanced class weights for the retained feature set. This change shows why the second tuning stage is useful: the best regularisation value can differ after feature selection and when the full training partition is used.

### Test performance

The final model achieved accuracy of 0.6747, spoiler precision of 0.4263, spoiler recall of 0.6860, spoiler F1 of 0.5259, macro-F1 of 0.6391, and average precision of 0.5114. The confusion matrix contains 56,735 true negatives, 27,863 false positives, 9,477 false negatives, and 20,708 true positives. The relatively high spoiler recall compared with precision is consistent with the use of balanced class weights: the model prioritises detecting spoiler reviews, but produces a substantial number of false positives.

### Takeaways and limitations

SentenceBERT provides dense semantic representations that can be combined with numeric and categorical metadata, while retaining interpretable block-level feature elimination. However, the current result is a frozen-encoder baseline rather than an end-to-end fine-tuned transformer. The experiment also uses a fixed random split, a small baseline tuning sample, a limited classifier grid, and a single SentenceBERT model. Therefore, the result should be treated as a reproducible baseline for comparison with TF-IDF and future transformer variants, not as the final model benchmark. Future work should compare multiple encoders, repeat evaluation across seeds or splits, measure runtime and memory usage, and investigate whether cue removal changes performance or exposes reliance on explicit spoiler language.
